# 13 — The mini-thesis: the whole method, small, on a free T4

**In one sentence:** before a week of GPU time, we run the WHOLE method once, small and free —
answer, keep the shortest correct answer, train a LoRA, test — to see if training works at all.

~~~text
100 MBPP+ TRAIN problems ─► model answers 4× (thinking ON) ─► keep shortest correct ─► train LoRA
                                                                  (25%, 50%, 100% of the examples)
100 OTHER MBPP+ TEST problems ◄───────────────────────────────────────────────┘
   6 ways: thinking OFF · ON · "think briefly" · ON+LoRA-25% · ON+LoRA-50% · ON+LoRA-100%
~~~

**Why MBPP+, not HumanEval?** HumanEval and LiveCodeBench are the final test set (DECISIONS #58).
Training on them, or choosing anything from their results, would make the final test unfair
(`CLAUDE.md` §4). MBPP+ was cut from the test set, so it is free to use. (DECISIONS #60)

**The 9 research questions, in short**

| | |
|---|---|
| Testing | Does shortest-correct LoRA training cut thinking on easy code, without losing accuracy? |
| Hypothesis | LoRA-100% uses ≤ 0.75× the tokens of thinking ON, losing ≤ 3 accuracy points |
| What we change | The way of answering (6 ways) and, for the LoRA, the amount of training data |
| What we measure | Accuracy (MBPP+'s own tests) and tokens per answer |
| Compared against | Thinking ON, thinking OFF, "think briefly" |
| Data | MBPP+: 100 train / 100 test, fixed-seed split, no overlap |
| Metric | Paired accuracy difference and token ratio, with error bars |
| Supports it | Rule R1 says YES (step 12) |
| Contradicts it | R1 says NO, or R3 says thinking OFF is just as good |

**Where we are:** `PROBLEM ✅ → GAP ✅ → QUESTION ✅ → HYPOTHESIS ✅ → EXPERIMENT ⬅ HERE (small version)`

**Before you run anything:** menu **Runtime → Change runtime type → T4 GPU**.
Run the cells in order. Steps 3, 5 and 6 are **safety nets** — if one fails, stop.
Estimated total: about 2–4 hours (not measured yet). Every step skips work it already finished.

## 1. Install what we need
**Problem:** Colab lacks our libraries. **Why:** nothing runs without them.
**In:** nothing. **Out:** installed packages + a GPU check.
**Why this way:** we install **Unsloth first**, because it pins its own library versions. Then
every step — answering AND training — uses the same versions, so the 6 ways stay comparable.
Qwen3.5 needs `transformers` version 5 or newer (Unsloth's Qwen3.5 guide).

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q evalplus datasets

In [ ]:
import torch, transformers
print("transformers", transformers.__version__,
      "OK" if int(transformers.__version__.split(".")[0]) >= 5 else "<-- need version 5+, STOP")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU  <-- STOP")
print("bfloat16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else "-")

## 2. Get the code, and a safe place to save answers
**Problem:** a free session can end at any second. **Why:** answers cost GPU time.
**In:** our GitHub repo. **Out:** the code, and a results folder on Google Drive.
**Why this way:** Drive outlives the session; every results file is only added to, and saved
after every batch. **If Colab disconnects:** run steps 1, 2, 4 and 6 again, then go on.

In [ ]:
import os, sys

REPO = "https://github.com/mahmudulhaquequdrati/stop-overthinking-thesis.git"
if not os.path.isdir("/content/thesis"):
    !git clone -q {REPO} /content/thesis
else:
    !cd /content/thesis && git pull -q
os.chdir("/content/thesis")
sys.path.insert(0, "/content/thesis/scripts")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/stop-overthinking/results/mini"
except Exception as e:
    print("No Google Drive (fine when testing locally):", e)
    OUT_DIR = "results/mini"
os.makedirs(OUT_DIR, exist_ok=True)
print("saving to:", OUT_DIR)

## 3. ⚠️ SAFETY NET — is the thinking counter correct?
**Problem:** Qwen puts its `<think>` marker in the prompt, not the answer. Old code counted
**0 thinking tokens on every answer** without crashing (DECISIONS #56).
**In:** only the tokenizer. **Out:** PASS or FAIL. **If anything says FAIL, stop here.**

In [ ]:
!python scripts/test_prompts.py

## 4. Get the MBPP+ problems, and check the grader
**Problem:** we need 100 train + 100 test problems with real tests, and a grader we can trust.
**Why:** if the grader is wrong, "shortest correct" picks wrong code and we train on it.
**In:** `evalplus/mbppplus` from Hugging Face. **Out:** `data/mbpp.json` and a grader check.
**Why this way:** the split is a fixed-seed shuffle, made before any answer exists. The grader
runs MBPP+'s **official solutions** first: they must all pass, or the grader is broken.

In [ ]:
!python scripts/mbpp_data.py
!python scripts/grade_mbpp.py --check-official 20

## 5. ⚠️ SAFETY NET — a 2-problem smoke test
**Problem:** a long run that is wrong from the first second is the most expensive mistake.
**In:** 2 train problems. **Out:** token counts.
👉 `thinking_on` must show thinking tokens in the **hundreds**; `thinking_off` must show **exactly 0**.

In [ ]:
import json
SMOKE = f"{OUT_DIR}/smoke-test.jsonl"
!rm -f "{SMOKE}"
for policy in ("thinking_on", "thinking_off"):
    !python scripts/gen_colab.py --policy {policy} --problems data/mbpp.json --source mbpp --split train \
        --n 2 --batch 2 --max-tokens 1024 --out "{SMOKE}"
rows = [json.loads(l) for l in open(SMOKE)]
for r in rows:
    print(f"{r['policy']:<14}{r['thinking_tokens']:>8} thinking {r['total_new_tokens']:>6} total")
ok = (all(r["thinking_tokens"] > 0 for r in rows if r["policy"] == "thinking_on") and
      all(r["thinking_tokens"] == 0 for r in rows if r["policy"] == "thinking_off"))
print("\n" + ("SMOKE TEST PASSED" if ok else "SMOKE TEST FAILED - stop, read step 3 again"))

## 6. ⚠️ SAFETY NET — does float16 give garbage?
**Problem:** Qwen was trained in **bfloat16**; the T4 cannot use it, so it uses **float16**,
which can overflow and make the model write garbage. **Why:** then every number here is wrong.
**In:** 10 **train** problems (never test ones), thinking ON, in float16 and in float32.
**Out:** pass counts, garbage counts, a verdict, and the number format for all later steps.
**Why this way:** float32 cannot overflow, so it is the honest reference. Rule, fixed before the
run (DECISIONS #59): float16 is OK if it passes at most 1 fewer and has no more garbage.

In [ ]:
FP = {d: f"{OUT_DIR}/fp-check-{d}.jsonl" for d in ("float16", "float32")}
for d, b in (("float16", 10), ("float32", 5)):
    !python scripts/gen_colab.py --policy thinking_on --problems data/mbpp.json --source mbpp --split train \
        --n 10 --batch {b} --dtype {d} --max-tokens 4096 --out "{FP[d]}"
    !python scripts/grade_mbpp.py --answers "{FP[d]}"

In [ ]:
import csv, json, re

# Garbage = one character 50+ times in a row (spaces and ----- lines allowed), or an answer
# that FINISHED thinking and still gave no code block. Both formats are judged the same way.
REPEAT = re.compile(r"([^\s\-=#*_])\1{49,}")

def summary(d):
    graded = list(csv.DictReader(open(FP[d].replace(".jsonl", "-graded.csv"))))
    rows = [json.loads(l) for l in open(FP[d])]
    garbage = sum(bool(REPEAT.search(r["raw_output"])) or
                  (not r["hit_limit"] and "```" not in r["answer_text"]) for r in rows)
    secs = sum(r["batch_seconds"] / r["batch_size"] for r in rows)
    return (sum(g["passed"] == "True" for g in graded), len(graded), garbage,
            sum(r["total_new_tokens"] for r in rows) / secs)

s16, s32 = summary("float16"), summary("float32")
for name, (passed, n, garbage, tps) in (("float16", s16), ("float32", s32)):
    print(f"{name}: passed {passed}/{n} | garbage answers {garbage} | ~{tps:.0f} tokens/s")
FP16_OK = s16[0] >= s32[0] - 1 and s16[2] <= s32[2]
DTYPE, BATCH = ("float16", 16) if FP16_OK else ("float32", 8)
print("\n" + ("float16 is SAFE -> everything below uses float16, 16 at a time." if FP16_OK else
              "float16 is NOT safe -> everything below uses float32, 8 at a time. Slower, still free."))

## 7. Step A — the model answers the 100 training problems, 4 times each
**Problem:** we need several correct answers per problem, some shorter than others.
**Why:** "keep the shortest correct" needs a choice. **In:** 100 train problems, thinking ON,
4 tries, the same 4,096-token limit as every other run. **Out:** 400 answers, graded.
**Why this way:** 4 tries is the plan (PLAN §8). A different fixed seed per try, so the tries differ.
This is the longest step: roughly 1–2 hours (estimate; watch the printed tokens/s).

In [ ]:
TRAIN_ANS = f"{OUT_DIR}/train-answers.jsonl"
!python scripts/gen_colab.py --policy thinking_on --problems data/mbpp.json --source mbpp --split train \
    --samples 4 --dtype {DTYPE} --batch {BATCH} --max-tokens 4096 --out "{TRAIN_ANS}"
!python scripts/grade_mbpp.py --answers "{TRAIN_ANS}"

## 8. Step B — keep the shortest correct answer per problem
**Problem:** turn 400 answers into one good, short example per problem.
**In:** the graded answers. **Out:** `train-set.jsonl` + **TARGET**.
**Why this way:** shortest correct, but not below half the median correct length (PLAN §7:
the very shortest answers can be lucky and hurt accuracy). Cut-off answers are never kept.

**TARGET** = how short the kept answers are compared with a normal correct answer
(0.70 = 30% shorter). A LoRA can't be expected to cut more than its examples show. Step 12 uses it.

In [ ]:
TRAIN_SET = f"{OUT_DIR}/train-set.jsonl"
!python scripts/make_train_set.py --answers "{TRAIN_ANS}" --out "{TRAIN_SET}"
TARGET = json.load(open(TRAIN_SET.replace(".jsonl", "-stats.json")))["target"]

## 9. Step C — train 3 LoRAs: the learning curve
**Problem:** you asked: *would more training data help?* One LoRA can't answer that.
**Why:** if a LoRA on 100% of the examples is clearly better than one on 50%, the curve is
still rising, and more data should help. If they are the same, it has levelled off.
**In:** the train set. **Out:** 3 LoRA folders on Drive (25%, 50%, 100% of the examples).
**Why this way:** the smaller sets sit **inside** the bigger ones, with the same settings and
epochs, so only the amount of data changes. Each takes minutes (the first one longer: Unsloth
compiles Qwen3.5's special layers first, which is slow on a T4).

In [ ]:
LORA = {f"lora{int(f * 100)}": f"{OUT_DIR}/lora/lora{int(f * 100)}" for f in (0.25, 0.5, 1.0)}
for (name, path), f in zip(LORA.items(), (0.25, 0.5, 1.0)):
    if os.path.exists(f"{path}/adapter_config.json"):
        print(name, "already trained, skipping"); continue
    !python scripts/train_lora.py --train "{TRAIN_SET}" --fraction {f} --out "{path}"

## 10. Step D — test all 6 ways on the 100 TEST problems
**Problem:** the real comparison. **Why:** this is the result.
**In:** the 100 test problems (the model and the LoRAs have never seen them), 1 try each.
**Out:** 6 answer files, graded.
**Why this way:** the same problems, the same 4,096 limit, the same settings and seed for every
way. Only the way of answering changes (`CLAUDE.md` §4). Each LoRA load is **checked**: if it
loaded empty, the script stops instead of silently using the base model (DECISIONS #57).

In [ ]:
WAYS = {"off": ("thinking_off", None), "on": ("thinking_on", None), "brief": ("brief", None),
        **{name: ("thinking_on", path) for name, path in LORA.items()}}
TEST = {name: f"{OUT_DIR}/test-{name}.jsonl" for name in WAYS}
for name, (policy, adapter) in WAYS.items():
    extra = f'--adapter "{adapter}"' if adapter else ""
    !python scripts/gen_colab.py --policy {policy} --label {name} {extra} --problems data/mbpp.json \
        --source mbpp --split test --dtype {DTYPE} --batch {BATCH} --max-tokens 4096 --out "{TEST[name]}"
    !python scripts/grade_mbpp.py --answers "{TEST[name]}"

## 11. Step E — compare, and read the verdicts
**Problem:** turn 600 graded answers into answers to our questions.
**In:** the 6 graded files + TARGET. **Out:** a table with error bars, the learning curve, 5 verdicts.
**Why this way:** every comparison is paired (same problems), and the verdict rules were
written **before** the run (DECISIONS #60).

In [ ]:
args = " ".join(f'{name}="{path.replace(".jsonl", "-graded.csv")}"' for name, path in TEST.items())
!python scripts/compare_mini.py --target {TARGET} {args}

## 12. You are here — what the verdicts mean

Decided **before** the run (DECISIONS #60):

~~~text
R1 YES (training works)          -> the method works on a T4. Go on to the real thesis run.
R1 NO                            -> training did not cut tokens, or it cost accuracy.
                                    Look at R5 before deciding anything.
R2 YES / NO                      -> does the LoRA beat simply asking to "think briefly"?
R3 YES (OFF is enough on easy)   -> easy problems can't show the value of thinking.
                                    The thesis must lean on MEDIUM problems.
R4 STILL IMPROVING               -> more training data should help: plan for more.
R4 FLAT                          -> more of the same data won't help: don't spend GPU time on it.
R5 reached the TARGET            -> to cut more, make SHORTER examples (8 tries, DECISIONS #27).
R5 not there yet                 -> more data or more epochs can still help.
~~~

**Honest limits:** 100 easy problems, 1 try each: differences smaller than the error bars are
noise. This is a **pilot**, not the thesis result, and MBPP+ is easier than our medium problems.

**Write the numbers down** in `results/2026-09-22-mini-thesis.md` and paste them into the chat.
A number that is not written down did not happen.